In [ ]:
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=20,
    executor_memory="16g",
    executor_cores=4,
    aws_profile="default",
)

## Rewrite Timeseries Data Files

Rewrites data files to apply partitioning and sort order from Migration 0002.

**Prerequisites:** Run `migrations/0002/add_partitioning_and_sort_order.sql` first.

Note: Only need `rewrite-all` once. The maintenance job will keep sorts up to date after.

In [ ]:
%%time
query = """
    CALL iceberg.system.rewrite_data_files(
        table => 'teehr.primary_timeseries',
        strategy => 'sort',
        sort_order => 'value_time ASC NULLS LAST, location_id ASC NULLS LAST',
        options => map('target-file-size-bytes', '536870912', 'rewrite-all', 'true')
    )
"""
spark.sql(query).show()

In [ ]:
%%time
query = """
    CALL iceberg.system.rewrite_data_files(
        table => 'teehr.secondary_timeseries',
        strategy => 'sort',
        sort_order => 'value_time ASC NULLS LAST, reference_time ASC NULLS LAST, location_id ASC NULLS LAST',
        options => map('target-file-size-bytes', '536870912', 'rewrite-all', 'true')
    )
"""
spark.sql(query).show()

In [86]:
spark.stop()